# Axiom Shopping Assistant EDA

This notebook documents the exploratory checks used to understand the Amazon product, category, and review files before loading them into PostgreSQL. It focuses on row retention, key distributions, category coverage, and review coverage after cleaning.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

RAW = ROOT / 'data' / 'raw'
CLEAN = ROOT / 'data' / 'cleaned'

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 20)

In [2]:
def count_rows(path):
    with open(path, 'rb') as f:
        return max(sum(1 for _ in f) - 1, 0)

row_counts = pd.DataFrame(
    {
        'raw_rows': {
            'products': count_rows(RAW / 'amazon_products.csv'),
            'reviews': count_rows(RAW / 'amazon_reviews.csv'),
            'categories': count_rows(RAW / 'amazon_categories.csv'),
            'brands': pd.NA,
            'users': pd.NA,
        },
        'clean_rows': {
            'products': count_rows(CLEAN / 'products.csv'),
            'reviews': count_rows(CLEAN / 'reviews.csv'),
            'categories': count_rows(CLEAN / 'categories.csv'),
            'brands': count_rows(CLEAN / 'brands.csv'),
            'users': count_rows(CLEAN / 'users.csv'),
        },
    }
)
row_counts['retention_pct'] = pd.NA
has_raw = row_counts['raw_rows'].notna()
row_counts.loc[has_raw, 'retention_pct'] = (
    row_counts.loc[has_raw, 'clean_rows'].astype(float)
    / row_counts.loc[has_raw, 'raw_rows'].astype(float)
    * 100
).round(2)
row_counts

            raw_rows  clean_rows  retention_pct
products     1426337     1426336         100.00
reviews       701528       11927           1.70
categories       248         248         100.00
brands         <NA>       26484            <NA>
users          <NA>       11902            <NA>

The product and category sources are effectively preserved by the cleaner. The review table is intentionally much smaller: only 11,927 of 701,528 raw reviews remain because most raw review ASINs are not present in the product corpus. This matters for the report because review sparsity is a source-data mismatch, not a database loading problem.

In [3]:
products = pd.read_csv(
    CLEAN / 'products.csv',
    usecols=['asin', 'price', 'list_price', 'stars', 'review_count', 'is_best_seller', 'category_id', 'brand_id'],
)
reviews = pd.read_csv(
    CLEAN / 'reviews.csv',
    usecols=['asin', 'user_id', 'rating', 'helpful_vote', 'verified_purchase', 'review_timestamp'],
    parse_dates=['review_timestamp'],
)
categories = pd.read_csv(CLEAN / 'categories.csv')
brands = pd.read_csv(CLEAN / 'brands.csv')

In [4]:
product_summary = pd.Series(
    {
        'unique_asins': products['asin'].nunique(),
        'priced_products': products['price'].notna().sum(),
        'price_missing_pct': products['price'].isna().mean() * 100,
        'price_mean': products['price'].mean(),
        'price_median': products['price'].median(),
        'price_max': products['price'].max(),
        'stars_mean': products['stars'].mean(),
        'stars_median': products['stars'].median(),
        'review_count_median': products['review_count'].fillna(0).median(),
        'review_count_max': products['review_count'].fillna(0).max(),
        'best_seller_count': products['is_best_seller'].fillna(False).astype(bool).sum(),
        'brand_nonnull_pct': products['brand_id'].notna().mean() * 100,
    }
).round(2)
product_summary

unique_asins              1426336.00
priced_products           1426336.00
price_missing_pct               0.00
price_mean                     43.38
price_median                   19.95
price_max                   19731.81
stars_mean                      4.00
stars_median                    4.40
review_count_median            0.00
review_count_max          346563.00
best_seller_count            8520.00
brand_nonnull_pct             30.07
dtype: float64

In [5]:
product_categories = products.merge(categories, on='category_id', how='left')
product_categories.groupby('category_name').size().sort_values(ascending=False).head(10)

category_name
Girls' Clothing       28619
Boys' Clothing        24660
Toys & Games          20846
Men's Shoes           19822
Women's Handbags      18994
Girls' Jewelry        18514
Men's Clothing        18258
Men's Accessories     17679
Women's Clothing      17393
Women's Jewelry       17005
dtype: int64

Product coverage is broad and clothing-heavy. Ratings are high overall, but the median `review_count` is zero, so routes that rank by both stars and review volume need to avoid treating unrated or lightly reviewed products as equally trustworthy.

In [6]:
review_summary = pd.Series(
    {
        'unique_users': reviews['user_id'].nunique(),
        'unique_reviewed_products': reviews['asin'].nunique(),
        'rating_mean': reviews['rating'].mean(),
        'rating_median': reviews['rating'].median(),
        'verified_pct': reviews['verified_purchase'].fillna(False).astype(bool).mean() * 100,
        'helpful_vote_mean': reviews['helpful_vote'].fillna(0).mean(),
        'helpful_vote_max': reviews['helpful_vote'].fillna(0).max(),
    }
).round(2)
review_summary

unique_users                11902.00
unique_reviewed_products      136.00
rating_mean                    4.22
rating_median                  5.00
verified_pct                  95.10
helpful_vote_mean              1.28
helpful_vote_max             646.00
dtype: float64

In [7]:
reviews['rating'].value_counts().sort_index()

1    1268
2     593
3     715
4    1053
5    8298
Name: count, dtype: int64

In [8]:
review_categories = (
    reviews[['asin']]
    .merge(products[['asin', 'category_id']], on='asin', how='left')
    .merge(categories, on='category_id', how='left')
)
review_categories.groupby('category_name').size().sort_values(ascending=False).head(10)

category_name
Hair Care Products                  4258
Makeup                              1794
Women's Accessories                 1159
Beauty Tools & Accessories           979
Industrial & Scientific              792
Travel Accessories                   438
Janitorial & Sanitation Supplies     415
Skin Care Products                   356
Foot, Hand & Nail Care Products      349
Home D\u00e9cor Products                 267
dtype: int64

The linked review subset is concentrated in beauty-related categories, especially Hair Care Products and Makeup. That explains why review-trend and credibility analytics are most meaningful for those categories and may be sparse elsewhere.

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

products['price'].clip(upper=250).hist(bins=50, ax=axes[0])
axes[0].set_title('Product prices clipped at $250')
axes[0].set_xlabel('price')
axes[0].set_ylabel('products')

products['stars'].round(1).value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Product star distribution')
axes[1].set_xlabel('stars')
axes[1].set_ylabel('products')

reviews['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[2])
axes[2].set_title('Linked review ratings')
axes[2].set_xlabel('rating')
axes[2].set_ylabel('reviews')

plt.tight_layout()

EDA takeaways: the normalized product catalog is large and stable, the linked review table is small but highly verified, and analytics that depend on review timestamps should use category fallbacks and empty states. These findings shaped the API validation, the UI empty states, and the optimized value/trending materialized views.